# ASG Airlines: evidence walkthrough

The source profile is recomputed from the workbook. Cleaned results come from the completed pipeline. Historical cloud evidence and Desktop verification are assessed separately.

In [1]:
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.kpi import calculate_kpis
from src.profile import profile_source, read_source

manifest_path = ROOT / "reports/run_manifest.json"
if not manifest_path.exists():
    raise RuntimeError("Run python -m src.pipeline with ASG_PII_PEPPER set before this notebook.")
manifest = json.loads(manifest_path.read_text())
assert manifest["status"] == "success"
with tempfile.TemporaryDirectory() as profile_dir:
    profile = profile_source(ROOT / "data/raw/UseCase_Airlines.xlsx", profile_dir)
assert profile["source_sha256"] == manifest["source_sha256"]
raw, _, _ = read_source(ROOT / "data/raw/UseCase_Airlines.xlsx")
pd.DataFrame({name: {"source_rows": item["rows"], "columns": len(item["columns"])}
              for name, item in profile["structure"].items()}).T

,source_rows,columns
flights,1020,7
payments,1000,4
bookings,1000,9
passengers,1039,9


SJ192 needs exactly one day added to arrival. The repaired flight lasts 300 minutes and arrives on the departure date.

In [2]:
before = raw["flights"].loc[raw["flights"].flight_id == "SJ192", ["flight_id", "departure_time", "arrival_time"]].assign(stage="source")
flights = pd.read_parquet(ROOT / "data/silver/flights.parquet")
after = flights.loc[flights.flight_id == "SJ192", ["flight_id", "departure_time", "arrival_time", "duration_minutes", "was_corrected", "is_overnight"]].assign(stage="silver")
pd.concat([before, after], ignore_index=True)

,flight_id,departure_time,arrival_time,stage,duration_minutes,was_corrected,is_overnight
0,SJ192,2026-04-19 18:45:42,2026-04-18 23:45:42,source,NaN,NaN,NaN
1,SJ192,2026-04-19 18:45:42,2026-04-19 23:45:42,silver,300.0,True,False


Excel stores 1,019 raw durations as time cells and one as a datetime representing a negative serial. Both must be decoded before reconciliation.

In [3]:
pd.DataFrame({"source_type": list(profile["structure"]["flights"]["source_types"]["duration"]),
              "rows": list(profile["structure"]["flights"]["source_types"]["duration"].values())})

,source_type,rows
0,time,1019
1,datetime,1


Missing amounts and the INVALID sentinel remain separate. Only valid amounts contribute to the payment total.

In [4]:
payment_types = profile["structure"]["payments"]["source_types"]["amount"]
payment_quality = pd.read_parquet(ROOT / "data/silver/payments.parquet").groupby("amount_quality").size()
display(pd.Series(payment_types, name="source_rows").to_frame())
payment_quality.to_frame("silver_rows")

,source_rows
float,913
NoneType,48
str,30
int,9


,silver_rows
amount_quality,
missing,48
non_numeric,30
valid,922


Duplicate flight keys repeat payments in a naive join. The warehouse uses unique flight keys and keeps each payment once.

In [5]:
pd.Series(profile["fanout"], name="measured_result").to_frame()

,measured_result
source_booking_rows,1000
booking_flight_left_join_rows,1032
booking_payment_inner_join_rows,1000
booking_payment_left_join_rows,1363
naive_three_way_inner_join_rows,1028
naive_three_way_left_join_rows,1404
true_payment_total,7385142.98
naive_three_way_total,7593758.92
inflation,208615.94


Survivorship selects one record by completeness, Aadhaar shape, email ordering, then source row. The audit shows ranks without contact values.

In [6]:
audit = pd.read_csv(ROOT / "reports/passenger_survivorship.csv")
example = audit.passenger_id.iloc[0]
audit.loc[audit.passenger_id == example]

,passenger_id,candidate_source_row,candidate_count,null_count,aadhaar_has_expected_length,candidate_rank,is_survivor,rule_used
0,P1034,37,2,0,True,1,True,lexicographic_email
1,P1034,36,2,0,True,2,False,lexicographic_email


The analytical passenger dimension contains a token and demographics. Raw names, contacts and DOB are absent.

In [7]:
passengers = pd.read_csv(ROOT / "data/gold/dim_passenger.csv")
passengers.loc[passengers.passenger_sk != -1].head(3)

,passenger_sk,passenger_token,age,age_band,gender
1,1,d43562e16e5eda2d8084e74b22f39f3cc71fa620547f79...,52.0,35-59,F
2,2,ae87489113611c71cfee954eb0dd2a553eb71d9905ebb3...,15.0,0-17,M
3,3,d175fb27268af1f2bcb8957d08900a7b6b96c65a466405...,72.0,60+,M


These KPIs use the verified final model: 1,003 real flights and all 1,000 bookings. Overnight and red-eye shares use real flights as their denominator. True delay is unavailable.

In [8]:
kpis = calculate_kpis(ROOT / "data/gold/asg_airlines.duckdb")
pd.Series(kpis, name="value").to_frame()

,value
flight_count,1003
avg_duration_minutes,164.672118
overnight_share,0.121635
red_eye_share,0.270189
booking_count,1000
payment_count,1000
gross_valid_payment_amount,7385142.98
confirmed_booking_payment_amount,2471402.04
cancellation_rate,0.314
confirmation_rate,0.32
